# WAXAL-NET fine-tuning on Colab

Works through `LANGUAGES` one at a time: train, evaluate, upload the results to the run's Hub repo.
Checkpoints go to the same `waxal-net-*` Hub repos as the Kaggle jobs, so a run started on either
platform resumes on the other. Give a language to one platform at a time (`jobs.py colab-claim`).

1. Runtime → Change runtime type → a GPU (L4 or A100 are much faster than T4).
2. Secrets (key icon, left bar) → add `HF_TOKEN` (a write token) and allow notebook access.
3. Run all. Keep the tab open: Colab Pro stops a session when its browser tab closes.
   If the session ends, run all again; training resumes from the last Hub checkpoint.

In [ ]:
LANGUAGES = ['wolaytta', 'sidama', 'oromo', 'amharic']
MODEL = "mms_300m"
FLEURS_LANGUAGES = ['amharic', 'fula', 'lingala', 'luganda', 'oromo', 'shona']
TRAIN_HOURS = 22  # stop training before Colab's 24 h session limit

In [ ]:
import os, subprocess, time
from pathlib import Path

SESSION_START = time.time()
REPO = Path("/content/asr_waxal")
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jdaltonll02/asr_waxal.git", str(REPO)], check=True)

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ.update(HF_HOME="/content/hf_cache", HF_DATASETS_DISABLE_PROGRESS_BARS="1", WANDB_DISABLED="true",
                  PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false",
                  PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
# The experiment configs write under /kaggle/working.
Path("/kaggle/working").mkdir(parents=True, exist_ok=True)
PY = "/tmp/waxal-venv/bin/python"
LOG_DIR = Path("/kaggle/working/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)


def run(*command, check=True):
    """Run a command from the repo root, streaming its output to the notebook and to LOG_DIR."""
    name = "_".join(Path(str(part)).stem for part in command[1:4])
    log_path = LOG_DIR / f"{time.strftime('%H%M%S')}_{name}.log"
    process = subprocess.Popen(list(command), cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1)
    with open(log_path, "w", encoding="utf-8") as log:
        for line in process.stdout:
            print(line, end="", flush=True)
            log.write(line)
            log.flush()
    if process.wait() and check:
        raise RuntimeError(f"{' '.join(map(str, command))} exited with {process.returncode}")
    return process.returncode


run("bash", f"{REPO}/kaggle/setup_env.sh")

# The configs' per-device batch of 2 fits a 15 GB T4 on clips up to ~57 s; bigger
# GPUs take more per step (accumulation drops so the effective batch stays 32).
import torch
memory_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
BATCH = ["--per-device-batch-size", "8" if memory_gb > 35 else "4"] if memory_gb > 20 else []
print(f"{torch.cuda.get_device_name(0)}, {memory_gb:.0f} GB: {BATCH or 'config batch size'}")

In [ ]:
import shutil
from huggingface_hub import HfApi

api = HfApi()
for LANGUAGE in LANGUAGES:
    print(f"===== {LANGUAGE} {MODEL} =====", flush=True)
    FLEURS = LANGUAGE in FLEURS_LANGUAGES
    RUN = f"finetune_{LANGUAGE}_{MODEL}"
    CONFIG = f"configs/experiment/{RUN}.yaml"
    CHECKPOINT = f"/kaggle/working/checkpoints/{RUN}/best_model"
    HUB = ["--hub-repo", f"waxal-net-{RUN}"] if os.environ.get("HF_TOKEN") else []
    run(PY, "-m", "src.train", "--config", CONFIG, "--deadline", str(SESSION_START + TRAIN_HOURS * 3600), *HUB, *BATCH)
    if Path(CHECKPOINT, "config.json").exists():
        run(PY, "-m", "src.evaluate", "--config", CONFIG, "--checkpoint", CHECKPOINT)
        if MODEL.startswith("whisper"):
            # Also score clips over 30 s in 30 s chunks; the paper doesn't say which it did.
            run(PY, "-m", "src.evaluate", "--config", CONFIG, "--checkpoint", CHECKPOINT, "--long-audio", "chunk30",
                "--output-dir", f"/kaggle/working/outputs/{RUN}_chunk30")
        if FLEURS:
            run(PY, "-m", "src.cross_domain_eval", "--config", f"configs/experiment/cross_domain_{LANGUAGE}_{MODEL}.yaml",
                "--checkpoint", CHECKPOINT)
    else:
        print("Training stopped at the session deadline. Run this notebook again to resume from the last checkpoint.")
    if not Path(CHECKPOINT, "config.json").exists():
        break  # stopped at the deadline; run all again to resume
    # Colab outputs are lost with the session; keep them next to the model.
    repo = f"{api.whoami()['name']}/waxal-net-{RUN}"
    for output in Path("/kaggle/working/outputs").glob(f"*_{LANGUAGE}_{MODEL}*"):
        api.upload_folder(repo_id=repo, folder_path=str(output), path_in_repo=f"outputs/{output.name}",
                          commit_message=f"{output.name} results")
    shutil.rmtree(f"/kaggle/working/checkpoints/{RUN}", ignore_errors=True)